In [1]:
#!/usr/bin/env python3
"""
macro_factors_complete.py
=========================
直接运行即可。下载所有宏观因子，滞后期处理干净，输出一个 CSV。

用法:
  1. pip install fredapi pandas numpy requests openpyxl xlrd
  2. 把下面的 FRED_API_KEY 换成你自己的（免费注册: fred.stlouisfed.org/docs/api/api_key/）
  3. python macro_factors_complete.py
  4. 得到 macro_factors_final.csv

时间对齐规则（Goyal & Welch 惯例）:
  df.loc[t] = 月末 t 时刻已知的信息 → 用于预测 r[t+1]
  Lag 0: 日频/周频市场数据, 月末即可得, 直接用
  Lag 1: 次月中旬才发布的宏观数据, .shift(1) 后 df.loc[t] = 第 t-1 月发布的值
"""

import warnings; warnings.filterwarnings("ignore")
import pandas as pd
import numpy as np
import requests
from io import BytesIO

try:
    from fredapi import Fred
except ImportError:
    raise SystemExit("请先运行: pip install fredapi")

# ─────────────────────────────────────────────────────────────────────────────
#  ★ 唯一需要修改的地方：填入你的 FRED API Key
#    免费申请: https://fred.stlouisfed.org/docs/api/api_key/
# ─────────────────────────────────────────────────────────────────────────────
FRED_API_KEY = "f199c39f506fa752d605eff11d0a65da"

START  = "1990-01-01"
END    = "2024-12-31"
OUTPUT = "macro_factors_final.csv"
# ─────────────────────────────────────────────────────────────────────────────

fred     = Fred(api_key=FRED_API_KEY)
results  = {}
log_rows = []


# ══════════════════════════════════════════════════════════════════════════════
#  工具函数
# ══════════════════════════════════════════════════════════════════════════════

def fetch(sid: str) -> pd.Series:
    s = fred.get_series(sid, observation_start=START, observation_end=END)
    s.index = pd.to_datetime(s.index)
    return s.dropna()


def to_me(s: pd.Series) -> pd.Series:
    """Resample 到月末（取每月最后一个可得观测值）。"""
    return s.resample("ME").last()


def store(col: str, s: pd.Series, lag: int, desc: str = ""):
    """施加滞后并存储。lag=0 → 不移位；lag=1 → .shift(1)。"""
    s = to_me(s).loc[START:END]
    if lag:
        s = s.shift(lag)
    results[col] = s
    log_rows.append({"变量": col, "lag(月)": lag, "描述": desc, "状态": "OK"})
    print(f"  ✓  {col:<28s} lag={lag}M   {desc}")


def fail(col: str, err, desc: str = ""):
    log_rows.append({"变量": col, "lag(月)": "—", "描述": desc, "状态": f"失败: {err}"})
    print(f"  ✗  {col:<28s} 失败: {err}")


def try_ids(ids: list) -> tuple:
    """按顺序尝试 FRED series ID，返回第一个成功的 (series, used_id)。"""
    for sid in ids:
        try:
            s = fetch(sid)
            print(f"       使用 FRED 系列: {sid}")
            return s, sid
        except Exception as e:
            print(f"       {sid} → {e}")
    raise ValueError(f"所有 ID 均失败: {ids}")


def log_yoy(s: pd.Series) -> pd.Series:
    """12个月 log 变化（年化%）。"""
    return np.log(s).diff(12) * 100


# ══════════════════════════════════════════════════════════════════════════════
#  BLOCK A — Lag 0: 金融市场数据（日频/周频，月末即可得）
# ══════════════════════════════════════════════════════════════════════════════
print("\n" + "═"*65)
print("  BLOCK A — Lag 0  (市场数据，无发布延迟)")
print("═"*65)

# A1. 联邦基金利率（水平）
try:
    store("fedfunds",      fetch("FEDFUNDS"),          lag=0, desc="联邦基金利率水平 (%)")
except Exception as e: fail("fedfunds", e)

# A2. 联邦基金利率（月度变化）→ 政策方向信号
try:
    store("d_fedfunds",    fetch("FEDFUNDS").diff(1),  lag=0, desc="联邦基金利率 MoM 变化 (pp)")
except Exception as e: fail("d_fedfunds", e)

# A3. 10年期 TIPS 实际收益率
try:
    store("real_rate_10y", fetch("DFII10"),             lag=0, desc="10Y TIPS 实际收益率 (%)")
except Exception as e: fail("real_rate_10y", e)

# A4. 10年期盈亏平衡通胀率（市场通胀预期）
try:
    store("bei_10y",       fetch("T10YIE"),             lag=0, desc="10Y 盈亏平衡通胀率 (%)")
except Exception as e: fail("bei_10y", e)

# A5. 美国高收益债 OAS（信用风险偏好）
#     BAMLH0A0HYM2: 因 ICE 版权限制，部分 FRED 账户无法访问
#     → 备用: Moody's BAA - GS10（公司债利差，信号几乎一致）
try:
    hy, _ = try_ids(["BAMLH0A0HYM2"])
    hy_me = to_me(hy).loc[START:END]
    coverage = 1 - hy_me.isna().mean()
    if coverage < 0.3:
        raise ValueError(f"数据覆盖率 {coverage:.0%} < 30%，切换备用方案")
    results["hy_oas"] = hy_me
    log_rows.append({"变量": "hy_oas", "lag(月)": 0,
                     "描述": "ICE BofA HY OAS", "状态": "OK"})
    print(f"  ✓  {'hy_oas':<28s} lag=0M   ICE BofA HY OAS (bp)")
except Exception:
    print("       BAMLH0A0HYM2 访问受限 → 改用 BAA-GS10 利差")
    try:
        baa  = to_me(fetch("BAA"))
        gs10 = to_me(fetch("GS10"))
        store("baa_gs10_spread", (baa - gs10),          lag=0,
              desc="Moody's BAA - 10Y 国债利差（HY OAS 替代）")
    except Exception as e: fail("baa_gs10_spread", e)

# A6. 美元指数（全球风险偏好代理变量）
try:
    store("dxy",    fetch("DTWEXBGS"),                  lag=0, desc="名义广义美元指数")
except Exception as e: fail("dxy", e)

# A7. VIX（隐含波动率）
try:
    store("vix",    fetch("VIXCLS"),                    lag=0, desc="CBOE VIX")
except Exception as e: fail("vix", e)

# A8. VIX 期限结构（3M/1M）→ >1 正向市场平静；<1 倒挂短期恐慌
try:
    vix3m = to_me(fetch("VXVCLS"))
    vix1m = to_me(fetch("VIXCLS"))
    store("vix_term_ratio", (vix3m / vix1m),            lag=0, desc="VIX 期限结构 (3M/1M)")
except Exception as e: fail("vix_term_ratio", e)

# A9. TED spread（银行间流动性压力，2023年后停更）
try:
    store("ted_spread", fetch("TEDRATE"),               lag=0, desc="TED Spread（2023年后停更）")
except Exception as e: fail("ted_spread", e)

# A10. 芝加哥 Fed 全国金融条件指数（负值 = 宽松）
try:
    store("nfci",   fetch("NFCI"),                      lag=0, desc="Chicago Fed NFCI")
except Exception as e: fail("nfci", e)

# A11. 10年期国债收益率
try:
    store("gs10",   fetch("GS10"),                      lag=0, desc="10Y 国债收益率 (%)")
except Exception as e: fail("gs10", e)

# A12. 期限利差（10Y - 3M）
try:
    tms = to_me(fetch("GS10")) - to_me(fetch("TB3MS"))
    store("term_spread", tms,                           lag=0, desc="期限利差: 10Y-3M (pp)")
except Exception as e: fail("term_spread", e)

# A13. 原材料/黄金比率（实体经济活动信号）
#      铜价: PCOPPUSDM (IMF) → 失败则用 WTI 原油/黄金比率替代
#      （铜价和油价都是全球增长预期的实物指标，方向高度一致）
print("\n  [A13] 原材料/黄金比率:")
cu_gold_ok = False
try:
    cu, cu_id = try_ids(["PCOPPUSDM", "PCOPP"])
    au = fetch("GOLDAMGBD228NLBM")
    store("cu_gold_log_ratio", np.log(cu / au),         lag=0,
          desc=f"Log(铜/金) ({cu_id})")
    cu_gold_ok = True
except Exception:
    pass

if not cu_gold_ok:
    print("       铜价系列不可用 → 改用 WTI原油/黄金比率（相似的增长信号）")
    try:
        oil = fetch("DCOILWTICO")
        au  = fetch("GOLDAMGBD228NLBM")
        store("oil_gold_log_ratio", np.log(oil / au),   lag=0,
              desc="Log(WTI原油/金) 增长信号（铜/金替代）")
    except Exception as e: fail("oil_gold_log_ratio", e)

# A14. 密歇根消费者信心指数（最终值月末前发布）
try:
    store("mich_sent", fetch("UMCSENT"),                lag=0, desc="密歇根消费者信心指数")
except Exception as e: fail("mich_sent", e)


# ══════════════════════════════════════════════════════════════════════════════
#  BLOCK B — Lag 1: 月度宏观数据（次月中旬才发布，.shift(1)）
#
#  发布日历（近似）:
#    ISM 制造业 PMI    → 次月第1个工作日
#    ISM 服务业 PMI    → 次月第3个工作日
#    NFP / 失业率      → 次月第1个周五
#    CB 先行指数 LEI   → 次月第3周
#    CPI               → 次月第2周
#    核心 PCE          → 次月最后一周
#    EPU / GPR         → 基于上月新闻构建，次月公布
# ══════════════════════════════════════════════════════════════════════════════
print("\n" + "═"*65)
print("  BLOCK B — Lag 1  (月度宏观，次月中旬发布，.shift(1))")
print("═"*65)

# B1. ISM 制造业 PMI（50为扩张/收缩分界线）
#     NAPM = 原始系列（可能已停更），INDPRO MoM 为最稳健替代
try:
    ism, _ = try_ids(["NAPM"])
    store("ism_mfg", ism,                               lag=1, desc="ISM 制造业 PMI")
except Exception:
    print("       NAPM 不可得 → 改用 INDPRO MoM% 作为制造业周期代理")
    try:
        store("ip_mom", fetch("INDPRO").pct_change()*100, lag=1,
              desc="工业生产 MoM%（ISM 制造业替代）")
    except Exception as e: fail("ip_mom", e)

# B2. ISM 服务业 PMI
#     NMFCI = 非制造业综合指数；NMFBAI = 商业活动指数；
#     RSAFS = 零售销售（最稳健替代）
try:
    ism_svc, _ = try_ids(["NMFCI", "NMFBAI"])
    store("ism_svc", ism_svc,                           lag=1, desc="ISM 服务业 PMI")
except Exception:
    print("       ISM 服务业不可得 → 改用零售销售额 MoM%")
    try:
        store("retail_sales_mom", fetch("RSAFS").pct_change()*100, lag=1,
              desc="零售销售 MoM%（ISM 服务业替代）")
    except Exception as e: fail("retail_sales_mom", e)

# B3. 非农就业人数月度变化
try:
    store("nfp_chg", fetch("PAYEMS").diff(1),           lag=1, desc="非农就业 MoM 变化（千人）")
except Exception as e: fail("nfp_chg", e)

# B4. 失业率
try:
    store("unrate", fetch("UNRATE"),                    lag=1, desc="失业率 (%)")
except Exception as e: fail("unrate", e)

# B5. 先行经济指标（Conference Board LEI）
#     USSLIND = CB LEI（首选）；USALOLITONOSTSAM = OECD CLI（备用）
try:
    lei, _ = try_ids(["USSLIND", "USALOLITONOSTSAM"])
    store("lei_chg", lei.diff(1),                       lag=1, desc="先行经济指标 MoM 变化")
except Exception as e: fail("lei_chg", e)

# B6. CPI 同比（log，年化%）
try:
    store("cpi_yoy", log_yoy(fetch("CPIAUCSL")),        lag=1, desc="CPI YoY log 变化 (%)")
except Exception as e: fail("cpi_yoy", e)

# B7. 核心 PCE 同比（Fed 最关注的通胀指标）
try:
    store("core_pce_yoy", log_yoy(fetch("PCEPILFE")),   lag=1, desc="核心 PCE YoY log 变化 (%)")
except Exception as e: fail("core_pce_yoy", e)

# B8. CPI 意外（实现值 - 过去12月均值，真正意外需 Bloomberg，此为代理）
try:
    cpi_mom = fetch("CPIAUCSL").pct_change() * 100
    store("cpi_surprise_proxy", cpi_mom - cpi_mom.rolling(12).mean(), lag=1,
          desc="CPI MoM 意外代理变量")
except Exception as e: fail("cpi_surprise_proxy", e)


# ══════════════════════════════════════════════════════════════════════════════
#  BLOCK C — 外部数据源（EPU & GPR），Lag 1
# ══════════════════════════════════════════════════════════════════════════════
print("\n" + "═"*65)
print("  BLOCK C — 外部数据源 (EPU, GPR)  Lag 1")
print("═"*65)

# C1. 经济政策不确定性指数（Baker-Bloom-Davis）
#     Excel 文件前几行是版权说明，需动态定位真正的 header 行
def parse_epu() -> pd.Series:
    url = "https://www.policyuncertainty.com/media/US_Policy_Uncertainty_Data.xlsx"
    raw = BytesIO(requests.get(url, timeout=30).content)

    # 先扫描所有行，找到含 "Year" 的 header 行
    df_scan = None
    for engine in ["openpyxl", "xlrd"]:
        try:
            raw.seek(0)
            df_scan = pd.read_excel(raw, header=None, engine=engine)
            break
        except Exception:
            continue
    if df_scan is None:
        raise ValueError("无法读取 EPU Excel 文件")

    hrow = None
    for i, row in df_scan.iterrows():
        if any(str(v).strip().lower() == "year" for v in row.values):
            hrow = i
            break
    if hrow is None:
        raise ValueError("未找到含 'Year' 的 header 行")

    # 以真正的 header 行重新读取
    df = None
    for engine in ["openpyxl", "xlrd"]:
        try:
            raw.seek(0)
            df = pd.read_excel(raw, header=hrow, engine=engine)
            break
        except Exception:
            continue
    if df is None:
        raise ValueError("以 header 行重新读取失败")

    df.columns = df.columns.str.strip()
    cm = {c.lower(): c for c in df.columns}
    yr = cm["year"]
    mo = cm["month"]
    # Three-Component Index（最常用的综合版本）
    vc = next((c for k, c in cm.items() if "three" in k), None)
    if vc is None:
        vc = [c for c in df.columns if c not in [yr, mo]][0]

    df = df[[yr, mo, vc]].copy()
    df[yr] = pd.to_numeric(df[yr], errors="coerce")
    df[mo] = pd.to_numeric(df[mo], errors="coerce")
    df = df.dropna(subset=[yr, mo])
    df["date"] = (
        pd.to_datetime({"year": df[yr].astype(int),
                        "month": df[mo].astype(int), "day": 1})
        + pd.offsets.MonthEnd(0)
    )
    return df.set_index("date")[vc].rename("epu").dropna()

try:
    epu = parse_epu().loc[START:END]
    store("epu", epu,                                   lag=1, desc="Baker-Bloom-Davis US EPU 指数")
except Exception as e:
    print(f"  policyuncertainty.com 获取失败: {e}")
    print("  尝试 FRED 备用版本 USEPUINDXD ...")
    try:
        store("epu", fetch("USEPUINDXD"),               lag=1, desc="EPU（FRED 日频备用）")
    except Exception as e2:
        fail("epu", e2, desc="Baker-Bloom-Davis EPU")

# C2. 地缘政治风险指数（Caldara & Iacoviello）
def parse_gpr() -> pd.Series:
    url = "https://www.matteoiacoviello.com/gpr_files/data_gpr_export.xls"
    raw = BytesIO(requests.get(url, timeout=30).content)

    df = None
    for engine in ["xlrd", "openpyxl"]:
        try:
            raw.seek(0)
            df = pd.read_excel(raw, engine=engine)
            break
        except Exception:
            continue
    if df is None:
        raise ValueError("无法读取 GPR Excel 文件")

    df.columns = df.columns.str.strip()
    cm = {c.lower(): c for c in df.columns}
    yr  = cm.get("year")
    mo  = cm.get("month")
    # GPR 综合指数（排除 threat/act 子指数）
    gpc = cm.get("gpr") or next(
        (c for k, c in cm.items()
         if k.startswith("gpr") and "threat" not in k and "act" not in k),
        None
    )
    if not all([yr, mo, gpc]):
        raise ValueError(
            f"列名识别失败。\n"
            f"  实际列名: {list(df.columns)}\n"
            f"  year={yr}, month={mo}, gpr={gpc}"
        )

    df = df[[yr, mo, gpc]].copy()
    df[yr] = pd.to_numeric(df[yr], errors="coerce")
    df[mo] = pd.to_numeric(df[mo], errors="coerce")
    df = df.dropna(subset=[yr, mo])
    df["date"] = (
        pd.to_datetime({"year": df[yr].astype(int),
                        "month": df[mo].astype(int), "day": 1})
        + pd.offsets.MonthEnd(0)
    )
    return df.set_index("date")[gpc].rename("gpr").dropna()

try:
    gpr = parse_gpr().loc[START:END]
    store("gpr", gpr,                                   lag=1, desc="Caldara-Iacoviello GPR 指数")
except Exception as e:
    print(f"  matteoiacoviello.com 获取失败: {e}")
    print("  → GPR 在 FRED 上没有备用。请手动下载:")
    print("    1. 访问 https://www.matteoiacoviello.com/gpr.htm")
    print("    2. 下载 data_gpr_export.xls，放到与本脚本相同的文件夹")
    print("    3. 重新运行脚本即可自动解析")
    fail("gpr", e, desc="Caldara-Iacoviello GPR")


# ══════════════════════════════════════════════════════════════════════════════
#  合并、清理、保存
# ══════════════════════════════════════════════════════════════════════════════
print("\n" + "═"*65)
print("  合并 & 保存")
print("═"*65)

df = pd.DataFrame(results)
df.index = pd.DatetimeIndex(df.index)
df.index.name = "date"
df = df.sort_index().loc[START:END]

# 保存主数据
df.to_csv(OUTPUT)

# 保存变量文档
pd.DataFrame(log_rows).to_csv("macro_factors_documentation.csv", index=False)

print(f"\n  输出文件:")
print(f"    {OUTPUT:<42s}  {df.shape[0]} 行 × {df.shape[1]} 列")
print(f"    macro_factors_documentation.csv          变量文档（含滞后期记录）")
print(f"\n  日期范围: {df.index[0].date()} → {df.index[-1].date()}")

# 缺失值汇总
miss = df.isna().sum()
miss = miss[miss > 0].sort_values(ascending=False)
if not miss.empty:
    print(f"\n  含缺失值的列（通常因为该系列起始日期较晚）:")
    for col, n in miss.items():
        pct = 100 * n / len(df)
        flag = "  ← 该系列在此期间不可得" if pct > 30 else ""
        print(f"    {col:<28s}  {n:4d} NaN ({pct:.0f}%){flag}")

# 成功/失败汇总
ok_cnt   = sum(1 for r in log_rows if r["状态"] == "OK")
fail_cnt = sum(1 for r in log_rows if r["状态"] != "OK")
print(f"\n  成功: {ok_cnt} 个变量   失败: {fail_cnt} 个变量")
if fail_cnt:
    print("  失败的变量:")
    for r in log_rows:
        if r["状态"] != "OK":
            print(f"    ✗  {r['变量']:<28s}  {r['状态']}")

print("\n  完成。\n")


═════════════════════════════════════════════════════════════════
  BLOCK A — Lag 0  (市场数据，无发布延迟)
═════════════════════════════════════════════════════════════════
  ✓  fedfunds                     lag=0M   联邦基金利率水平 (%)
  ✓  d_fedfunds                   lag=0M   联邦基金利率 MoM 变化 (pp)
  ✓  real_rate_10y                lag=0M   10Y TIPS 实际收益率 (%)
  ✓  bei_10y                      lag=0M   10Y 盈亏平衡通胀率 (%)
       使用 FRED 系列: BAMLH0A0HYM2
  ✓  hy_oas                       lag=0M   ICE BofA HY OAS (bp)
  ✓  dxy                          lag=0M   名义广义美元指数
  ✓  vix                          lag=0M   CBOE VIX
  ✓  vix_term_ratio               lag=0M   VIX 期限结构 (3M/1M)
  ✓  ted_spread                   lag=0M   TED Spread（2023年后停更）
  ✓  nfci                         lag=0M   Chicago Fed NFCI
  ✓  gs10                         lag=0M   10Y 国债收益率 (%)
  ✓  term_spread                  lag=0M   期限利差: 10Y-3M (pp)

  [A13] 原材料/黄金比率:
       使用 FRED 系列: PCOPPUSDM
       铜价系列不可用 → 改用 WTI原油/黄金比率（相似的增长信号）
  ✗  o

In [5]:
#!/usr/bin/env python3
"""
macro_factors_complete.py
=========================
直接运行即可。下载所有宏观因子，滞后期处理干净，输出一个 CSV。

用法:
  1. pip install fredapi pandas numpy requests openpyxl xlrd
  2. 把下面的 FRED_API_KEY 换成你自己的（免费注册: fred.stlouisfed.org/docs/api/api_key/）
  3. python macro_factors_complete.py
  4. 得到 macro_factors_final.csv

时间对齐规则（Goyal & Welch 惯例）:
  df.loc[t] = 月末 t 时刻已知的信息 → 用于预测 r[t+1]
  Lag 0: 日频/周频市场数据, 月末即可得, 直接用
  Lag 1: 次月中旬才发布的宏观数据, .shift(1) 后 df.loc[t] = 第 t-1 月发布的值
"""

import warnings; warnings.filterwarnings("ignore")
import pandas as pd
import numpy as np
import requests
from io import BytesIO

try:
    from fredapi import Fred
except ImportError:
    raise SystemExit("请先运行: pip install fredapi")

# ─────────────────────────────────────────────────────────────────────────────
#  ★ 唯一需要修改的地方：填入你的 FRED API Key
#    免费申请: https://fred.stlouisfed.org/docs/api/api_key/
# ─────────────────────────────────────────────────────────────────────────────
FRED_API_KEY = "f199c39f506fa752d605eff11d0a65da"

START = "1990-01-01"
END   = "2024-12-31"
OUTPUT = "macro_factors_final.csv"
# ─────────────────────────────────────────────────────────────────────────────

fred = Fred(api_key=FRED_API_KEY)
results  = {}   # col -> pd.Series
log_rows = []   # for documentation


# ══════════════════════════════════════════════════════════════════════════════
#  工具函数
# ══════════════════════════════════════════════════════════════════════════════

def fetch(sid: str) -> pd.Series:
    s = fred.get_series(sid, observation_start=START, observation_end=END)
    s.index = pd.to_datetime(s.index)
    return s.dropna()


def to_me(s: pd.Series) -> pd.Series:
    """Resample to month-end (last available obs in each month)."""
    return s.resample("ME").last()


def store(col: str, s: pd.Series, lag: int, desc: str = ""):
    """Apply lag and store. lag=0 → no shift; lag=1 → .shift(1)."""
    s = to_me(s).loc[START:END]
    if lag:
        s = s.shift(lag)
    results[col] = s
    log_rows.append({"col": col, "lag": lag, "desc": desc, "status": "OK"})
    print(f"  ✓  {col:<26s} lag={lag}M   {desc}")


def fail(col: str, err, desc: str = ""):
    log_rows.append({"col": col, "lag": "—", "desc": desc, "status": f"FAIL: {err}"})
    print(f"  ✗  {col:<26s} ERROR: {err}")


def try_ids(ids: list) -> pd.Series:
    """Try FRED series IDs in order; return first success."""
    for sid in ids:
        try:
            s = fetch(sid)
            print(f"       used FRED series: {sid}")
            return s
        except Exception as e:
            print(f"       {sid} → {e}")
    raise ValueError(f"All IDs failed: {ids}")


def log_yoy(s: pd.Series) -> pd.Series:
    return np.log(s).diff(12) * 100


# ══════════════════════════════════════════════════════════════════════════════
#  BLOCK A — Lag 0: 金融市场数据（日频/周频，月末即可得）
# ══════════════════════════════════════════════════════════════════════════════
print("\n" + "═"*65)
print("  BLOCK A — Lag 0  (市场数据，无发布延迟)")
print("═"*65)

# A1. 联邦基金利率（水平）
try:
    store("fedfunds",   fetch("FEDFUNDS"),               lag=0, desc="Fed Funds Rate level (%)")
except Exception as e: fail("fedfunds", e)

# A2. 联邦基金利率（月度变化）→ 政策方向信号
try:
    store("d_fedfunds", fetch("FEDFUNDS").diff(1),       lag=0, desc="Fed Funds Rate MoM change (pp)")
except Exception as e: fail("d_fedfunds", e)

# A3. 10年期 TIPS 实际收益率
try:
    store("real_rate_10y", fetch("DFII10"),              lag=0, desc="10Y TIPS Real Yield (%)")
except Exception as e: fail("real_rate_10y", e)

# A4. 10年期盈亏平衡通胀率（市场通胀预期）
try:
    store("bei_10y",    fetch("T10YIE"),                 lag=0, desc="10Y Breakeven Inflation (%)")
except Exception as e: fail("bei_10y", e)

# A5. 美国高收益债 OAS（信用风险偏好）
try:
    store("hy_oas",     fetch("BAMLH0A0HYM2"),           lag=0, desc="ICE BofA US HY OAS (bp)")
except Exception as e: fail("hy_oas", e)

# A6. 美元指数（全球风险偏好代理）
try:
    store("dxy",        fetch("DTWEXBGS"),               lag=0, desc="Nominal Broad Dollar Index")
except Exception as e: fail("dxy", e)

# A7. VIX（隐含波动率）
try:
    store("vix",        fetch("VIXCLS"),                 lag=0, desc="CBOE VIX")
except Exception as e: fail("vix", e)

# A8. VIX 期限结构（3M/1M）> 1 = 正向 = 市场平静；< 1 = 倒挂 = 短期恐慌
try:
    vix3m = to_me(fetch("VXVCLS"))   # 3-month VIX
    vix1m = to_me(fetch("VIXCLS"))
    vix_ts = (vix3m / vix1m).rename("vix_term_ratio")
    store("vix_term_ratio", vix_ts,                      lag=0, desc="VIX Term Structure (3M/1M)")
except Exception as e: fail("vix_term_ratio", e)

# A9. TED spread（银行间流动性压力，2023年后停更）
try:
    store("ted_spread",  fetch("TEDRATE"),               lag=0, desc="TED Spread (discontinued 2023)")
except Exception as e: fail("ted_spread", e)

# A10. 芝加哥 Fed 全国金融条件指数（负值 = 宽松）
try:
    store("nfci",       fetch("NFCI"),                   lag=0, desc="Chicago Fed NFCI")
except Exception as e: fail("nfci", e)

# A11. 10年期国债收益率
try:
    store("gs10",       fetch("GS10"),                   lag=0, desc="10Y Treasury Yield (%)")
except Exception as e: fail("gs10", e)

# A12. 期限利差（10Y - 3M）
try:
    tms = to_me(fetch("GS10")) - to_me(fetch("TB3MS"))
    store("term_spread", tms,                            lag=0, desc="Term Spread: 10Y - 3M (pp)")
except Exception as e: fail("term_spread", e)

# A13. 铜/金比率（"Dr. Copper"，实体经济活动信号）
#      FRED铜价系列按优先级尝试，金价用 LBMA 官方价格
try:
    cu  = try_ids(["PCOPPUSDM",   # IMF Global Copper Price USD/ton (月度，首选)
                   "PCOPP",       # World Bank Commodity: Copper
                   "WPU102"])     # PPI for Copper Products (代理)
    au  = fetch("GOLDAMGBD228NLBM")
    store("cu_gold_log_ratio", np.log(cu / au),          lag=0, desc="Log(Copper/Gold) growth signal")
except Exception as e: fail("cu_gold_log_ratio", e, desc="Log(Copper/Gold) growth signal")

# A14. 密歇根消费者信心（最终值月末前发布）
try:
    store("mich_sent",  fetch("UMCSENT"),                lag=0, desc="U. Michigan Consumer Sentiment")
except Exception as e: fail("mich_sent", e)


# ══════════════════════════════════════════════════════════════════════════════
#  BLOCK B — Lag 1: 月度宏观数据（次月中旬才发布，.shift(1)）
#
#  发布日历（近似）:
#    ISM 制造业 PMI    → 次月第1个工作日
#    ISM 服务业 PMI    → 次月第3个工作日
#    NFP / 失业率      → 次月第1个周五
#    CB 先行指数 LEI   → 次月第3周
#    CPI               → 次月第2周
#    核心 PCE          → 次月最后一周
#    EPU / GPR         → 基于上月新闻构建，次月公布
# ══════════════════════════════════════════════════════════════════════════════
print("\n" + "═"*65)
print("  BLOCK B — Lag 1  (月度宏观，次月中旬发布，.shift(1))")
print("═"*65)

# B1. ISM 制造业 PMI（50分界线）
#     NAPM = 原始系列（可能已停更），INDPRO MoM 作为最稳健的替代
try:
    ism = try_ids(["NAPM"])          # ISM Purchasing Managers Index
    store("ism_mfg", ism,            lag=1, desc="ISM Manufacturing PMI")
except Exception:
    print("     NAPM 不可得，改用 INDPRO MoM% 作为制造业周期代理")
    try:
        store("ip_mom",
              fetch("INDPRO").pct_change() * 100, lag=1,
              desc="Industrial Production MoM% (ISM proxy)")
    except Exception as e: fail("ip_mom", e)

# B2. ISM 服务业 PMI
#     NMFCI = Non-Manufacturing Composite；NMFBAI = Business Activity；
#     RSAFS = Advance Retail Sales（最稳健替代）
try:
    ism_svc = try_ids(["NMFCI", "NMFBAI"])
    store("ism_svc", ism_svc,        lag=1, desc="ISM Services PMI")
except Exception:
    print("     ISM服务业不可得，改用零售销售额 MoM 作为服务消费代理")
    try:
        store("retail_sales_mom",
              fetch("RSAFS").pct_change() * 100, lag=1,
              desc="Advance Retail Sales MoM% (ISM Svc proxy)")
    except Exception as e: fail("retail_sales_mom", e)

# B3. 非农就业人数变化（月度）
try:
    store("nfp_chg",
          fetch("PAYEMS").diff(1),   lag=1, desc="Nonfarm Payrolls MoM change (000s)")
except Exception as e: fail("nfp_chg", e)

# B4. 失业率
try:
    store("unrate",   fetch("UNRATE"), lag=1, desc="Unemployment Rate (%)")
except Exception as e: fail("unrate", e)

# B5. 先行经济指标（Conference Board LEI）
#     USSLIND = CB LEI（首选）；USALOLITONOSTSAM = OECD CLI（备用）
try:
    lei = try_ids(["USSLIND", "USALOLITONOSTSAM"])
    store("lei_chg", lei.diff(1),    lag=1, desc="Leading Econ Index MoM change")
except Exception as e: fail("lei_chg", e)

# B6. CPI 同比（log，年化%）
try:
    store("cpi_yoy",  log_yoy(fetch("CPIAUCSL")),
                                     lag=1, desc="CPI YoY log change (%)")
except Exception as e: fail("cpi_yoy", e)

# B7. 核心 PCE 同比（Fed最关注的通胀指标）
try:
    store("core_pce_yoy",
          log_yoy(fetch("PCEPILFE")), lag=1, desc="Core PCE YoY log change (%)")
except Exception as e: fail("core_pce_yoy", e)

# B8. CPI 意外（实现值 - 过去12月均值，真正意外需Bloomberg，此为代理）
try:
    cpi_mom = fetch("CPIAUCSL").pct_change() * 100
    store("cpi_surprise_proxy",
          cpi_mom - cpi_mom.rolling(12).mean(),
                                     lag=1, desc="CPI MoM surprise proxy")
except Exception as e: fail("cpi_surprise_proxy", e)


# ══════════════════════════════════════════════════════════════════════════════
#  BLOCK C — 外部数据源（EPU & GPR）
# ══════════════════════════════════════════════════════════════════════════════
print("\n" + "═"*65)
print("  BLOCK C — 外部数据源 (EPU, GPR)  Lag 1")
print("═"*65)

# C1. 经济政策不确定性指数（Baker-Bloom-Davis）
#     Excel前几行是版权说明，需动态定位真正的 header 行
def parse_epu() -> pd.Series:
    url = "https://www.policyuncertainty.com/media/US_Policy_Uncertainty_Data.xlsx"
    raw = BytesIO(requests.get(url, timeout=30).content)

    # 扫描找 header 行（含 "Year" 的那一行）
    df_scan = None
    for engine in ["openpyxl", "xlrd"]:
        try:
            raw.seek(0)
            df_scan = pd.read_excel(raw, header=None, engine=engine)
            break
        except Exception:
            continue
    if df_scan is None:
        raise ValueError("无法读取 EPU Excel 文件")

    hrow = None
    for i, row in df_scan.iterrows():
        if any(str(v).strip().lower() == "year" for v in row.values):
            hrow = i; break
    if hrow is None:
        raise ValueError("未找到含 'Year' 的 header 行")

    df = None
    for engine in ["openpyxl", "xlrd"]:
        try:
            raw.seek(0)
            df = pd.read_excel(raw, header=hrow, engine=engine)
            break
        except Exception:
            continue
    if df is None:
        raise ValueError("以 header 行重新读取失败")

    df.columns = df.columns.str.strip()
    cm = {c.lower(): c for c in df.columns}
    yr = cm["year"]; mo = cm["month"]
    # Three-Component Index（最常用的综合版本）
    vc = next((c for k, c in cm.items() if "three" in k), None)
    if vc is None:
        vc = [c for c in df.columns if c not in [yr, mo]][0]

    df = df[[yr, mo, vc]].copy()
    df[yr] = pd.to_numeric(df[yr], errors="coerce")
    df[mo] = pd.to_numeric(df[mo], errors="coerce")
    df = df.dropna(subset=[yr, mo])
    df["date"] = (pd.to_datetime({"year": df[yr].astype(int),
                                  "month": df[mo].astype(int), "day": 1})
                  + pd.offsets.MonthEnd(0))
    return df.set_index("date")[vc].rename("epu").dropna()

try:
    epu = parse_epu().loc[START:END]
    store("epu", epu, lag=1, desc="Baker-Bloom-Davis US EPU Index")
except Exception as e:
    print(f"  policyuncertainty.com 获取失败: {e}")
    print("  尝试 FRED 备用版本 USEPUINDXD ...")
    try:
        store("epu", fetch("USEPUINDXD"), lag=1, desc="EPU (FRED daily fallback)")
    except Exception as e2:
        fail("epu", e2, desc="Baker-Bloom-Davis EPU")

# C2. 地缘政治风险指数（Caldara & Iacoviello）
def parse_gpr() -> pd.Series:
    url = "https://www.matteoiacoviello.com/gpr_files/data_gpr_export.xls"
    raw = BytesIO(requests.get(url, timeout=30).content)

    df = None
    for engine in ["xlrd", "openpyxl"]:
        try:
            raw.seek(0)
            df = pd.read_excel(raw, engine=engine)
            break
        except Exception:
            continue
    if df is None:
        raise ValueError("无法读取 GPR Excel 文件")

    df.columns = df.columns.str.strip()
    cm = {c.lower(): c for c in df.columns}
    yr  = cm.get("year")
    mo  = cm.get("month")
    # GPR 综合指数（排除 threat/act 子指数）
    gpc = cm.get("gpr") or next(
        (c for k, c in cm.items()
         if k.startswith("gpr") and "threat" not in k and "act" not in k),
        None
    )
    if not all([yr, mo, gpc]):
        raise ValueError(f"列名识别失败，实际列名: {list(df.columns)}")

    df = df[[yr, mo, gpc]].copy()
    df[yr] = pd.to_numeric(df[yr], errors="coerce")
    df[mo] = pd.to_numeric(df[mo], errors="coerce")
    df = df.dropna(subset=[yr, mo])
    df["date"] = (pd.to_datetime({"year": df[yr].astype(int),
                                  "month": df[mo].astype(int), "day": 1})
                  + pd.offsets.MonthEnd(0))
    return df.set_index("date")[gpc].rename("gpr").dropna()

try:
    gpr = parse_gpr().loc[START:END]
    store("gpr", gpr, lag=1, desc="Caldara-Iacoviello GPR Index")
except Exception as e:
    print(f"  matteoiacoviello.com 获取失败: {e}")
    try:
        store("gpr", fetch("GPRI"), lag=1, desc="GPR (FRED fallback)")
    except Exception as e2:
        fail("gpr", e2, desc="Caldara-Iacoviello GPR")


# ══════════════════════════════════════════════════════════════════════════════
#  合并、清理、保存
# ══════════════════════════════════════════════════════════════════════════════
print("\n" + "═"*65)
print("  合并 & 保存")
print("═"*65)

df = pd.DataFrame(results)
df.index = pd.DatetimeIndex(df.index)
df.index.name = "date"
df = df.sort_index().loc[START:END]

# 保存主数据
df.to_csv(OUTPUT)

# 保存文档
doc = pd.DataFrame(log_rows)
doc.to_csv("macro_factors_documentation.csv", index=False)

print(f"\n  输出文件:")
print(f"    {OUTPUT:<40s}  {df.shape[0]} 行 × {df.shape[1]} 列")
print(f"    macro_factors_documentation.csv        变量文档（含滞后期记录）")
print(f"\n  日期范围: {df.index[0].date()} → {df.index[-1].date()}")

# 缺失值汇总
miss = df.isna().sum()
miss = miss[miss > 0].sort_values(ascending=False)
if not miss.empty:
    print(f"\n  含缺失值的列（通常是因为该系列起始日期较晚）:")
    for col, n in miss.items():
        print(f"    {col:<26s}  {n:4d} NaN ({100*n/len(df):.0f}%)")

# 成功/失败汇总
ok_cnt   = sum(1 for r in log_rows if r["status"] == "OK")
fail_cnt = sum(1 for r in log_rows if r["status"] != "OK")
print(f"\n  成功: {ok_cnt} 个变量   失败: {fail_cnt} 个变量")
if fail_cnt:
    print(f"  失败的变量:")
    for r in log_rows:
        if r["status"] != "OK":
            print(f"    {r['col']:<26s}  {r['status']}")

print("\n  完成。\n")


═════════════════════════════════════════════════════════════════
  BLOCK A — Lag 0  (市场数据，无发布延迟)
═════════════════════════════════════════════════════════════════
  ✓  fedfunds                   lag=0M   Fed Funds Rate level (%)
  ✓  d_fedfunds                 lag=0M   Fed Funds Rate MoM change (pp)
  ✓  real_rate_10y              lag=0M   10Y TIPS Real Yield (%)
  ✓  bei_10y                    lag=0M   10Y Breakeven Inflation (%)
  ✓  hy_oas                     lag=0M   ICE BofA US HY OAS (bp)
  ✓  dxy                        lag=0M   Nominal Broad Dollar Index
  ✓  vix                        lag=0M   CBOE VIX
  ✓  vix_term_ratio             lag=0M   VIX Term Structure (3M/1M)
  ✓  ted_spread                 lag=0M   TED Spread (discontinued 2023)
  ✓  nfci                       lag=0M   Chicago Fed NFCI
  ✓  gs10                       lag=0M   10Y Treasury Yield (%)
  ✓  term_spread                lag=0M   Term Spread: 10Y - 3M (pp)
       used FRED series: PCOPPUSDM
  ✗  cu_gold_lo

In [ ]:
#!/usr/bin/env python3
"""
fix_broken_series.py  — 修复5个失败变量
"""
import warnings; warnings.filterwarnings("ignore")
import pandas as pd, numpy as np, requests
from io import BytesIO
from fredapi import Fred

FRED_API_KEY = "YOUR_FRED_API_KEY_HERE"
START, END = "1990-01-01", "2024-12-31"
fred = Fred(api_key=FRED_API_KEY)
results = {}

def fetch(sid):
    s = fred.get_series(sid, observation_start=START, observation_end=END)
    s.index = pd.to_datetime(s.index); return s.dropna()

def me(s): return s.resample("ME").last()

def store(col, s, lag):
    s = me(s).loc[START:END]
    if lag: s = s.shift(lag)
    results[col] = s
    print(f"  OK  {col:<25s} lag={lag}M")

def try_ids(ids):
    for sid in ids:
        try:
            s = fetch(sid); print(f"     used: {sid}"); return s
        except Exception as e:
            print(f"     {sid} failed: {e}")
    raise ValueError(f"all failed: {ids}")

# ── Fix 1: Copper/Gold ──────────────────────────────────────────────────────
print("\n[1] Copper/Gold Ratio")
try:
    cu = try_ids(["PCOPPUSDM", "PCOPP", "WPU102"])
    au = me(fetch("GOLDAMGBD228NLBM"))
    store("cu_gold_log_ratio", np.log(cu/au), lag=0)
except Exception as e:
    print(f"  FAIL: {e}")

# ── Fix 2: ISM Manufacturing ────────────────────────────────────────────────
print("\n[2] ISM Manufacturing PMI")
try:
    store("ism_mfg", try_ids(["NAPM"]), lag=1)
except Exception:
    try:
        store("ip_mom", fetch("INDPRO").pct_change()*100, lag=1)
        print("     (INDPRO MoM% used as proxy — rename col if needed)")
    except Exception as e:
        print(f"  FAIL: {e}")

# ── Fix 3: ISM Services ─────────────────────────────────────────────────────
print("\n[3] ISM Services PMI")
try:
    store("ism_svc", try_ids(["NMFCI","NMFBAI"]), lag=1)
except Exception:
    try:
        store("retail_sales_mom", fetch("RSAFS").pct_change()*100, lag=1)
        print("     (RSAFS MoM% used as proxy — rename col if needed)")
    except Exception as e:
        print(f"  FAIL: {e}")

# ── Fix 4: EPU ──────────────────────────────────────────────────────────────
print("\n[4] EPU — Baker-Bloom-Davis")
def parse_epu():
    url = "https://www.policyuncertainty.com/media/US_Policy_Uncertainty_Data.xlsx"
    raw = BytesIO(requests.get(url, timeout=30).content)
    # scan for header row
    for engine in ["openpyxl","xlrd"]:
        try:
            raw.seek(0)
            scan = pd.read_excel(raw, header=None, engine=engine)
            break
        except: pass
    hrow = next(
        (i for i,row in scan.iterrows()
         if any(str(v).strip().lower()=="year" for v in row.values)),
        None
    )
    if hrow is None: raise ValueError("header row not found")
    raw.seek(0)
    for engine in ["openpyxl","xlrd"]:
        try:
            raw.seek(0); df = pd.read_excel(raw, header=hrow, engine=engine); break
        except: pass
    df.columns = df.columns.str.strip()
    cm = {c.lower():c for c in df.columns}
    yr, mo = cm["year"], cm["month"]
    # three-component index = 3rd column after year/month
    vc = next((c for k,c in cm.items() if "three" in k), None) or \
         [c for c in df.columns if c not in [yr,mo]][0]
    df = df[[yr,mo,vc]].dropna(subset=[yr,mo])
    df[yr] = pd.to_numeric(df[yr], errors="coerce")
    df[mo] = pd.to_numeric(df[mo], errors="coerce")
    df = df.dropna()
    df["date"] = pd.to_datetime({"year":df[yr].astype(int),"month":df[mo].astype(int),"day":1}) + pd.offsets.MonthEnd(0)
    return df.set_index("date")[vc].rename("epu").dropna()

try:
    store("epu", parse_epu().loc[START:END], lag=1)
except Exception as e:
    print(f"  policyuncertainty.com failed: {e}")
    try:
        store("epu", fetch("USEPUINDXD"), lag=1)   # FRED daily EPU fallback
    except Exception as e2:
        print(f"  FRED EPU fallback also failed: {e2}")

# ── Fix 5: GPR ──────────────────────────────────────────────────────────────
print("\n[5] GPR — Caldara-Iacoviello")
def parse_gpr():
    url = "https://www.matteoiacoviello.com/gpr_files/data_gpr_export.xls"
    raw = BytesIO(requests.get(url, timeout=30).content)
    df = None
    for engine in ["xlrd","openpyxl"]:
        try:
            raw.seek(0); df = pd.read_excel(raw, engine=engine); break
        except: pass
    if df is None: raise ValueError("cannot read GPR Excel")
    df.columns = df.columns.str.strip()
    cm = {c.lower():c for c in df.columns}
    yr  = cm.get("year")
    mo  = cm.get("month")
    # GPR overall: avoid "threat"/"act" sub-indexes
    gpc = cm.get("gpr") or next(
        (c for k,c in cm.items() if k.startswith("gpr") and "threat" not in k and "act" not in k),
        None
    )
    if not all([yr,mo,gpc]):
        raise ValueError(f"columns: {list(df.columns)}")
    df = df[[yr,mo,gpc]].dropna(subset=[yr,mo])
    df[yr] = pd.to_numeric(df[yr], errors="coerce")
    df[mo] = pd.to_numeric(df[mo], errors="coerce")
    df = df.dropna()
    df["date"] = pd.to_datetime({"year":df[yr].astype(int),"month":df[mo].astype(int),"day":1}) + pd.offsets.MonthEnd(0)
    return df.set_index("date")[gpc].rename("gpr").dropna()

try:
    store("gpr", parse_gpr().loc[START:END], lag=1)
except Exception as e:
    print(f"  matteoiacoviello.com failed: {e}")
    try:
        store("gpr", fetch("GPRI"), lag=1)          # FRED GPR fallback
    except Exception as e2:
        print(f"  FRED GPR fallback also failed: {e2}")

# ── Save ─────────────────────────────────────────────────────────────────────
print("\n── Saving ──────────────────────────────────────────────────────────────")
if results:
    df_fix = pd.DataFrame(results)
    df_fix.index = pd.DatetimeIndex(df_fix.index)
    df_fix.index.name = "date"
    df_fix.sort_index().loc[START:END].to_csv("macro_factors_fix.csv")
    print(f"  Saved macro_factors_fix.csv  ({df_fix.shape[1]} cols)")
    print()
    print("  Merge with original:")
    print("    main = pd.read_csv('macro_factors_lagged.csv', index_col='date', parse_dates=True)")
    print("    fix  = pd.read_csv('macro_factors_fix.csv',    index_col='date', parse_dates=True)")
    print("    for col in fix.columns: main[col] = fix[col]")
    print("    main.to_csv('macro_factors_final.csv')")

In [1]:
#!/usr/bin/env python3
"""
download_macro_factors.py
=========================
Download macro factors for MDN weighting with clean publication-lag handling.

Convention (Goyal & Welch style):
  X[t] predicts r[t+1]
  X[t] = information available at END of month t (= start of month t+1)

Publication lag applied via .shift(n) AFTER downloading raw series:

  Lag 0  → daily/weekly financial market data; end-of-month value used directly
            (FEDFUNDS, real rates, spreads, VIX, DXY, NFCI, Michigan final)

  Lag +1 → monthly macro released ~mid next month via .shift(1)
            (ISM PMI, NFP, UNRATE, LEI, CPI, Core PCE, EPU, GPR)

Sources:
  FRED   → fredapi (free key at fred.stlouisfed.org)
  EPU    → Baker-Bloom-Davis (policyuncertainty.com)
  GPR    → Caldara-Iacoviello (matteoiacoviello.com)

Requirements:
  pip install fredapi pandas numpy requests openpyxl xlrd wrds
"""

import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import requests
from io import BytesIO

try:
    from fredapi import Fred
except ImportError:
    raise ImportError("Run: pip install fredapi")

# ─────────────────────────────────────────────────────────────────────────────
# CONFIG  — put your FRED API key here (free: fred.stlouisfed.org/docs/api/)
# ─────────────────────────────────────────────────────────────────────────────
FRED_API_KEY = "f199c39f506fa752d605eff11d0a65da"
START = "1990-01-01"
END   = "2024-12-31"

fred = Fred(api_key=FRED_API_KEY)

OK  = "✓"
ERR = "✗"

In [2]:

# ─────────────────────────────────────────────────────────────────────────────
# HELPERS
# ─────────────────────────────────────────────────────────────────────────────

def fetch(series_id: str) -> pd.Series:
    """Download FRED series and return with DatetimeIndex."""
    s = fred.get_series(series_id, observation_start=START, observation_end=END)
    s.index = pd.to_datetime(s.index)
    return s.dropna()


def to_month_end(s: pd.Series) -> pd.Series:
    """Resample to month-end (last available observation in each month)."""
    return s.resample("ME").last()


def log_yoy(s: pd.Series) -> pd.Series:
    """12-month log change (annualised pct, *100)."""
    return np.log(s).diff(12) * 100


def log_mom(s: pd.Series) -> pd.Series:
    return np.log(s).diff(1) * 100


results = {}  # col_name → pd.Series (already lag-adjusted, month-end index)
log_rows = []


def store(col: str, s: pd.Series, desc: str, lag: int):
    """Apply publication lag and store."""
    s = s.loc[START:END]
    if lag > 0:
        s = s.shift(lag)
    results[col] = s
    note = f"+{lag}M shift" if lag else "no shift"
    log_rows.append({"Variable": col, "Lag applied": lag, "Description": desc, "Note": note})
    print(f"  {OK}  {col:<22s}  lag={lag}M  {desc}")


def fail(col: str, err):
    log_rows.append({"Variable": col, "Lag applied": "—", "Description": "FAILED", "Note": str(err)})
    print(f"  {ERR}  {col:<22s}  ERROR: {err}")

In [3]:

# ─────────────────────────────────────────────────────────────────────────────
# BLOCK A — LAG 0: Financial/market data (daily/weekly → month-end)
#           Information fully observable by END of month t
# ─────────────────────────────────────────────────────────────────────────────
print("\n── Block A: Lag 0 (market data, no publication delay) ─────────────────")

# A1. Federal Funds Rate (level)
try:
    ff = to_month_end(fetch("FEDFUNDS"))
    store("fedfunds", ff, "Fed Funds Rate, level (%)", lag=0)
except Exception as e: fail("fedfunds", e)

# A2. ΔFed Funds Rate (MoM change) — policy direction signal
try:
    d_ff = to_month_end(fetch("FEDFUNDS")).diff(1)
    store("d_fedfunds", d_ff, "Fed Funds Rate, MoM change (pp)", lag=0)
except Exception as e: fail("d_fedfunds", e)

# A3. 10-Year TIPS Real Yield (daily)
try:
    real = to_month_end(fetch("DFII10"))
    store("real_rate_10y", real, "10Y TIPS Real Yield (%)", lag=0)
except Exception as e: fail("real_rate_10y", e)

# A4. 10-Year Breakeven Inflation (daily) — market inflation expectations
try:
    bei = to_month_end(fetch("T10YIE"))
    store("bei_10y", bei, "10Y Breakeven Inflation Rate (%)", lag=0)
except Exception as e: fail("bei_10y", e)

# A5. HY Option-Adjusted Spread — credit risk appetite
try:
    hy = to_month_end(fetch("BAMLH0A0HYM2"))
    store("hy_oas", hy, "ICE BofA US HY OAS (bp)", lag=0)
except Exception as e: fail("hy_oas", e)

# A6. Nominal Broad Dollar Index
try:
    dxy = to_month_end(fetch("DTWEXBGS"))
    store("dxy", dxy, "Nominal Broad Dollar Index", lag=0)
except Exception as e: fail("dxy", e)

# A7. VIX (daily)
try:
    vix = to_month_end(fetch("VIXCLS"))
    store("vix", vix, "CBOE VIX", lag=0)
except Exception as e: fail("vix", e)

# A8. VIX Term Structure: VIX3M / VIX  (>1 = contango = calm; <1 = backwardation = stressed)
try:
    vix3m = to_month_end(fetch("VXVCLS"))   # CBOE 3-month VIX
    vix1m = to_month_end(fetch("VIXCLS"))
    vix_ts = (vix3m / vix1m).rename("vix_term_ratio")
    store("vix_term_ratio", vix_ts, "VIX Term Structure (3M/1M)", lag=0)
except Exception as e: fail("vix_term_ratio", e)

# A9. TED Spread (LIBOR - T-bill; discontinued 2023, keep for history)
try:
    ted = to_month_end(fetch("TEDRATE"))
    store("ted_spread", ted, "TED Spread (pp, discontinued 2023)", lag=0)
except Exception as e: fail("ted_spread", e)

# A10. Chicago Fed National Financial Conditions Index (weekly → month-end)
try:
    nfci = to_month_end(fetch("NFCI"))
    store("nfci", nfci, "Chicago Fed NFCI (neg=loose)", lag=0)
except Exception as e: fail("nfci", e)

# A11. 10Y Treasury Yield
try:
    gs10 = to_month_end(fetch("GS10"))
    store("gs10", gs10, "10Y Treasury Yield (%)", lag=0)
except Exception as e: fail("gs10", e)

# A12. Term Spread: 10Y - 3M
try:
    tb3m = to_month_end(fetch("TB3MS"))
    gs10_ = to_month_end(fetch("GS10"))
    tms = (gs10_ - tb3m).rename("term_spread")
    store("term_spread", tms, "Term Spread: 10Y − 3M (pp)", lag=0)
except Exception as e: fail("term_spread", e)

# A13. Copper/Gold Ratio — "Dr. Copper" growth signal (log)
try:
    # IMF copper price (USD/metric ton), monthly
    cu = to_month_end(fetch("PCOPPUSDM"))
    # LBMA gold fix (USD/troy oz), daily → month-end
    au = to_month_end(fetch("GOLDAMGBD228NLBM"))
    cu_au = np.log(cu / au).rename("cu_gold_log_ratio")
    store("cu_gold_log_ratio", cu_au, "Log Copper/Gold Ratio (growth signal)", lag=0)
except Exception as e: fail("cu_gold_log_ratio", e)

# A14. Michigan Consumer Sentiment — final reading released last Friday of month
#      → available by month-end → lag 0
try:
    mich = to_month_end(fetch("UMCSENT"))
    store("mich_sent", mich, "U. Michigan Consumer Sentiment", lag=0)
except Exception as e: fail("mich_sent", e)


── Block A: Lag 0 (market data, no publication delay) ─────────────────
  ✓  fedfunds                lag=0M  Fed Funds Rate, level (%)
  ✓  d_fedfunds              lag=0M  Fed Funds Rate, MoM change (pp)
  ✓  real_rate_10y           lag=0M  10Y TIPS Real Yield (%)
  ✓  bei_10y                 lag=0M  10Y Breakeven Inflation Rate (%)
  ✓  hy_oas                  lag=0M  ICE BofA US HY OAS (bp)
  ✓  dxy                     lag=0M  Nominal Broad Dollar Index
  ✓  vix                     lag=0M  CBOE VIX
  ✓  vix_term_ratio          lag=0M  VIX Term Structure (3M/1M)
  ✓  ted_spread              lag=0M  TED Spread (pp, discontinued 2023)
  ✓  nfci                    lag=0M  Chicago Fed NFCI (neg=loose)
  ✓  gs10                    lag=0M  10Y Treasury Yield (%)
  ✓  term_spread             lag=0M  Term Spread: 10Y − 3M (pp)
  ✗  cu_gold_log_ratio       ERROR: Bad Request.  The series does not exist.
  ✓  mich_sent               lag=0M  U. Michigan Consumer Sentiment


In [4]:



# ─────────────────────────────────────────────────────────────────────────────
# BLOCK B — LAG +1: Monthly macro released ~mid next month
#           Month-t value NOT available until month t+1 → .shift(1)
#
#           Release calendar (approximate):
#             ISM Mfg PMI    → 1st business day of month t+1
#             NFP / UNRATE   → 1st Friday of month t+1
#             CB LEI         → ~3rd week of month t+1
#             CPI            → ~2nd week of month t+1
#             Core PCE       → last week of month t+1
#             EPU / GPR      → constructed from prior month's news → t+1
# ─────────────────────────────────────────────────────────────────────────────
print("\n── Block B: Lag +1 (monthly macro, published ~mid next month) ──────────")

# B1. ISM Manufacturing PMI  (50 = expansion/contraction boundary)
try:
    ism = to_month_end(fetch("NAPM"))
    store("ism_mfg", ism, "ISM Manufacturing PMI", lag=1)
except Exception as e: fail("ism_mfg", e)

# B2. ISM Services / Non-Manufacturing Composite
try:
    ism_svc = to_month_end(fetch("NMFCI"))
    store("ism_svc", ism_svc, "ISM Services (Non-Mfg) PMI", lag=1)
except Exception as e: fail("ism_svc", e)

# B3. Nonfarm Payrolls — MoM change (thousands)
try:
    nfp = to_month_end(fetch("PAYEMS")).diff(1)
    store("nfp_chg", nfp, "Nonfarm Payrolls MoM change (000s)", lag=1)
except Exception as e: fail("nfp_chg", e)

# B4. Unemployment Rate
try:
    unrate = to_month_end(fetch("UNRATE"))
    store("unrate", unrate, "Unemployment Rate (%)", lag=1)
except Exception as e: fail("unrate", e)

# B5. Conference Board Leading Economic Index — MoM change is the standard signal
try:
    lei = to_month_end(fetch("USSLIND"))
    store("lei_chg", lei.diff(1), "CB Leading Econ Index MoM change", lag=1)
except Exception as e:
    # Fallback: OECD CLI for USA
    try:
        cli = to_month_end(fetch("USALOLITONOSTSAM"))
        store("lei_chg", cli.diff(1), "OECD CLI MoM change (LEI fallback)", lag=1)
    except Exception as e2: fail("lei_chg", e2)

# B6. CPI — YoY log change (annualised %)
try:
    cpi = to_month_end(fetch("CPIAUCSL"))
    store("cpi_yoy", log_yoy(cpi), "CPI YoY log chg (%)", lag=1)
except Exception as e: fail("cpi_yoy", e)

# B7. Core PCE — Fed's preferred inflation gauge, YoY log change
try:
    pce = to_month_end(fetch("PCEPILFE"))
    store("core_pce_yoy", log_yoy(pce), "Core PCE YoY log chg (%)", lag=1)
except Exception as e: fail("core_pce_yoy", e)

# B8. CPI Surprise = CPI MoM realised − prior month's consensus
#     Approximation: deviation from trailing 12M average (proper version needs Bloomberg)
try:
    cpi_mom = to_month_end(fetch("CPIAUCSL")).pct_change(1) * 100
    cpi_surp = cpi_mom - cpi_mom.rolling(12).mean()
    store("cpi_surprise_proxy", cpi_surp,
          "CPI MoM surprise proxy (realised − 12M mean)", lag=1)
except Exception as e: fail("cpi_surprise_proxy", e)


── Block B: Lag +1 (monthly macro, published ~mid next month) ──────────
  ✗  ism_mfg                 ERROR: Bad Request.  The series does not exist.
  ✗  ism_svc                 ERROR: Bad Request.  The series does not exist.
  ✓  nfp_chg                 lag=1M  Nonfarm Payrolls MoM change (000s)
  ✓  unrate                  lag=1M  Unemployment Rate (%)
  ✓  lei_chg                 lag=1M  CB Leading Econ Index MoM change
  ✓  cpi_yoy                 lag=1M  CPI YoY log chg (%)
  ✓  core_pce_yoy            lag=1M  Core PCE YoY log chg (%)
  ✓  cpi_surprise_proxy      lag=1M  CPI MoM surprise proxy (realised − 12M mean)


In [5]:


# ─────────────────────────────────────────────────────────────────────────────
# BLOCK C — External sources (EPU, GPR)
# ─────────────────────────────────────────────────────────────────────────────
print("\n── Block C: External sources (EPU, GPR) ───────────────────────────────")

def download_epu() -> pd.Series:
    """Baker-Bloom-Davis US Economic Policy Uncertainty Index."""
    url = "https://www.policyuncertainty.com/media/US_Policy_Uncertainty_Data.xlsx"
    r = requests.get(url, timeout=30)
    r.raise_for_status()
    df = pd.read_excel(BytesIO(r.content))
    df.columns = df.columns.str.strip()
    # Find the Three-Component index column
    val_col = next(
        (c for c in df.columns if "Three" in str(c) or "three" in str(c)),
        df.columns[2]
    )
    df["date"] = pd.to_datetime({
        "year":  df["Year"].astype(int),
        "month": df["Month"].astype(int),
        "day":   1
    }) + pd.offsets.MonthEnd(0)
    return df.set_index("date")[val_col].rename("epu").dropna()

try:
    epu = download_epu().loc[START:END]
    store("epu", epu, "Baker-Bloom-Davis US EPU Index", lag=1)
except Exception as e: fail("epu", e)


def download_gpr() -> pd.Series:
    """Caldara & Iacoviello Geopolitical Risk Index."""
    url = "https://www.matteoiacoviello.com/gpr_files/data_gpr_export.xls"
    r = requests.get(url, timeout=30)
    r.raise_for_status()
    df = pd.read_excel(BytesIO(r.content))
    df.columns = df.columns.str.strip().str.lower()
    df["date"] = pd.to_datetime({
        "year":  df["year"].astype(int),
        "month": df["month"].astype(int),
        "day":   1
    }) + pd.offsets.MonthEnd(0)
    return df.set_index("date")["gpr"].rename("gpr").dropna()

try:
    gpr = download_gpr().loc[START:END]
    store("gpr", gpr, "Caldara-Iacoviello Geopolitical Risk Index", lag=1)
except Exception as e: fail("gpr", e)


── Block C: External sources (EPU, GPR) ───────────────────────────────
  ✗  epu                     ERROR: invalid literal for int() with base 10: "Source: 'Measuring Economic Policy Uncertainty' by Scott Baker, Nicholas Bloom and Steven J. Davis at www.PolicyUncertainty.com.  These data can be used freely with attribution to the authors, the pa
  ✗  gpr                     ERROR: 'year'


In [2]:




# ─────────────────────────────────────────────────────────────────────────────
# BLOCK D — WRDS: Philadelphia Fed Real-Time Data (optional, best practice)
#           Provides ALFRED-style real-time vintages → cleanest OOS lag handling
#           Uncomment if you have WRDS access
# ─────────────────────────────────────────────────────────────────────────────
print("\n── Block D: WRDS Philadelphia Fed Real-Time Data ───────────────────────")
try:
    import wrds
    db = wrds.Connection()

    # Real-time CPI (Philly Fed vintage)
    # This gives the value ACTUALLY available at each vintage date
    # Table: pfed.realtime (check WRDS data dictionary for exact table name)
    rt_cpi = db.raw_sql("""
        SELECT vintage_date, date, value
        FROM pfed.realtime
        WHERE series = 'CPIAUCSL'
        ORDER BY vintage_date, date
    """)
    # Use same-month or next-month vintage (whichever is the first release)
    # This replaces the mechanical .shift(1) with actual release timing
    db.close()
except Exception as e:
    print(f"  ✗ WRDS real-time data: {e}")


── Block D: WRDS Philadelphia Fed Real-Time Data ───────────────────────
Loading library list...
Done
  ✗ WRDS real-time data: (psycopg2.errors.UndefinedTable) relation "pfed.realtime" does not exist
LINE 3:         FROM pfed.realtime
                     ^

[SQL: 
        SELECT vintage_date, date, value
        FROM pfed.realtime
        WHERE series = 'CPIAUCSL'
        ORDER BY vintage_date, date
    ]
(Background on this error at: https://sqlalche.me/e/20/f405)


In [ ]:


# ─────────────────────────────────────────────────────────────────────────────
# MERGE & CLEAN
# ─────────────────────────────────────────────────────────────────────────────
print("\n── Merging all series ──────────────────────────────────────────────────")

df = pd.DataFrame(results)
df.index = pd.DatetimeIndex(df.index)
df.index.name = "date"
df = df.sort_index().loc[START:END]

# ─────────────────────────────────────────────────────────────────────────────
# SANITY CHECK: Verify lag alignment
# The value stored in row[t] must use data released BEFORE end of month t.
#
# For lag-0 variables: value = month-t market close  ✓ available
# For lag-1 variables: .shift(1) → row[t] = month t-1 value,
#                      which is released early month t  ✓ available
# ─────────────────────────────────────────────────────────────────────────────
lag_map = pd.DataFrame(log_rows).set_index("Variable")

print(f"\n{'='*65}")
print(f"  Final DataFrame: {df.shape[0]} months × {df.shape[1]} factors")
print(f"  Date range    : {df.index[0].date()} → {df.index[-1].date()}")
print(f"\n  Missing values per column:")
miss = df.isna().sum()
for col, n in miss[miss > 0].items():
    print(f"    {col:<25s}  {n:4d} NaN  "
          f"({100*n/len(df):.0f}%,  series starts later)")

# ─────────────────────────────────────────────────────────────────────────────
# SAVE
# ─────────────────────────────────────────────────────────────────────────────
df.to_csv("/mnt/user-data/outputs/macro_factors_lagged.csv")
lag_map.to_csv("/mnt/user-data/outputs/macro_factors_lag_reference.csv")

print(f"\n  ✓ macro_factors_lagged.csv         → main factor table")
print(f"  ✓ macro_factors_lag_reference.csv  → lag documentation")
print(f"{'='*65}\n")

if __name__ == "__main__":
    print("Done.")